# Practice 2.2 - Word Embedding

Objectives:
- Download the multilingual WikiNews corpus and keep **English text only**.
- Train a **Word2Vec** model with `gensim`.
- Evaluate the trained embeddings on the **MSR word analogy** benchmark.

Data sources:
- WikiNews multilingual corpus: https://github.com/PrimerAI/WikiNews-multilingual/blob/main/multilingual_wikinews.jsonl
- MSR analogy test: https://drive.google.com/file/d/1tYcY7GtaYm3m5VOioZRHDrK1Ob8z2nxl/view?usp=sharing


## Table of Contents

1. [Section 1: Setup](#Section-1:-Setup)
   - Import libraries and define paths
   - Download the corpus and the analogy file
   - Extract English text and tokenize it
2. [Section 2.1: Train Word2Vec (Skip-gram)](#Section-2.1:-Train-Word2Vec-(Skip-gram))
   - Configure and train the Skip-gram model
   - Inspect nearest neighbors for a sample word
3. [Section 2.2: Train Word2Vec (CBOW)](#Section-2.2:-Train-Word2Vec-(CBOW))
   - Configure and train the CBOW model
   - Inspect nearest neighbors for a sample word
4. [Section 3.1: Evaluate Word Analogies (Skip-gram)](#Section-3.1:-Evaluate-Word-Analogies-(Skip-gram))
   - Load the MSR analogy questions
   - Evaluate Skip-gram analogy accuracy
5. [Section 3.2: Evaluate Word Analogies (CBOW)](#Section-3.2:-Evaluate-Word-Analogies-(CBOW))
   - Evaluate CBOW analogy accuracy
   - Review random correct and incorrect predictions


## Section 1: Setup

In [ ]:
# Installing required packages
!pip install gensim tqdm

In [ ]:
from pathlib import Path
import json
import os

import pandas as pd

from gensim.models import Word2Vec
from gensim.utils import simple_preprocess

In [ ]:
# Local data files expected under the notebook's data directory.
# No download step is required.
required_filenames = ["multilingual_wikinews.jsonl", "msr.csv"]

In [ ]:
data_dir = Path.cwd() / "data"
data_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
corpus_path = data_dir / "multilingual_wikinews.jsonl"
msr_path = data_dir / "msr.csv"

print(f"Data directory: {data_dir.resolve()}")

In [ ]:
# Verify the required local corpus and analogy files are available.
required_paths = [corpus_path, msr_path]
missing_paths = [path for path in required_paths if not path.exists()]

if missing_paths:
    missing_message = "\n".join(str(path.resolve()) for path in missing_paths)
    raise FileNotFoundError(f"Missing required local data files:\n{missing_message}")

for path in required_paths:
    print(f"Found local file: {path.resolve()}")

In [ ]:
# Extract English text from WikiNews for corpus inspection and model training.
def extract_english_corpus(jsonl_path: Path):
    english_segments = []
    english_articles = []

    with jsonl_path.open("r", encoding="utf-8") as input_file:
        for line in input_file:
            record = json.loads(line)

            # TODO: Clean the title and text parts, combine non-empty segments, and add them to english_segments.
            ############ WRITE YOUR CODE HERE ############
            english_articles.append(" ".join(segments))

    return english_segments, english_articles


english_segments, english_articles = extract_english_corpus(corpus_path)

print(f"English training segments: {len(english_segments):,}")
print(f"English articles         : {len(english_articles):,}")
print()
print("Sample article:")
print(english_articles[0][:500])

In [ ]:
# Tokenize English segments into Word2Vec-ready token sequences.
# TODO: Convert each English segment into a token list, then keep only sequences with at least 3 tokens.
# Hint: apply the length filter with if len(tokens) >= 3, and use min_len=2 and max_len=30 in your tokenizer.
############ WRITE YOUR CODE HERE ############

token_count = sum(len(tokens) for tokens in sentences)
unique_tokens = len({token for tokens in sentences for token in tokens})

print(f"Tokenized sequences: {len(sentences):,}")
print(f"Total tokens       : {token_count:,}")
print(f"Unique tokens      : {unique_tokens:,}")
print(f"Example sequence   : {sentences[0][:20]}")

## Section 2.1: Train Word2Vec (Skip-gram)

In [ ]:
# Train Skip-gram Word2Vec embeddings on the tokenized English corpus.
# TODO: Define the Skip-gram Word2Vec training configuration with vector_size, window, min_count, workers, sg, and epochs.
############ WRITE YOUR CODE HERE ############

skipgram_model_path = data_dir / "skipgram.model"
# TODO: Train the Skip-gram Word2Vec model using the tokenized sentences and config.
############ WRITE YOUR CODE HERE ############
skipgram_model.save(str(skipgram_model_path))

print(f"Skip-gram vocabulary size: {len(skipgram_model.wv):,}")

In [ ]:
# Quick sanity check: nearest neighbors from the Skip-gram model.
probe_word = "government"

skipgram_neighbors_df = pd.DataFrame(
    skipgram_model.wv.most_similar(probe_word, topn=10),
    columns=["neighbor", "cosine_similarity"],
)
skipgram_neighbors_df

## Section 2.2: Train Word2Vec (CBOW)

In [ ]:
# Train CBOW Word2Vec embeddings on the tokenized English corpus.
# TODO: Define the CBOW Word2Vec training configuration with vector_size, window, min_count, workers, sg, epochs.
############ WRITE YOUR CODE HERE ############

cbow_model_path = data_dir / "cbow.model"
# TODO: Train the CBOW Word2Vec model using the tokenized sentences and config.
############ WRITE YOUR CODE HERE ############
cbow_model.save(str(cbow_model_path))

print(f"CBOW vocabulary size: {len(cbow_model.wv):,}")

In [ ]:
# Quick sanity check: nearest neighbors from the CBOW model.
probe_word = "government"

cbow_neighbors_df = pd.DataFrame(
    cbow_model.wv.most_similar(probe_word, topn=10),
    columns=["neighbor", "cosine_similarity"],
)
cbow_neighbors_df

## Section 3.1: Evaluate Word Analogies (Skip-gram)

In [ ]:
# Load and normalize MSR analogy questions for evaluation.
msr_df = pd.read_csv(msr_path)
msr_df = msr_df[["type", "word1", "word2", "word3", "target"]].copy()

for column in ["word1", "word2", "word3", "target"]:
    msr_df[column] = msr_df[column].astype(str).str.strip().str.lower()

print(f"MSR analogy questions: {len(msr_df):,}")
msr_df.head()

In [ ]:
# Define a helper function for analogy prediction.
def solve_analogy(model: Word2Vec, word1: str, word2: str, word3: str, topn: int = 10) -> str | None:
    """Predict the missing word in an analogy with vector arithmetic.

    Args:
        model: Trained Word2Vec model.
        word1: First word in the source pair.
        word2: Second word in the source pair.
        word3: First word in the target pair.
        topn: Number of candidate neighbors to inspect.

    Returns:
        The best predicted word that is not already in the prompt, or `None`.
    """
    # TODO: Find the best analogy candidate using vector arithmetic and skip words already present in the prompt.
    ############ WRITE YOUR CODE HERE ############

In [ ]:
# Define a helper function for analogy evaluation reporting.
def evaluate_word_analogies(model: Word2Vec, analogy_df: pd.DataFrame):
    """Evaluate analogy accuracy on the subset covered by the vocabulary.

    Args:
        model: Trained Word2Vec model.
        analogy_df: DataFrame with `word1`, `word2`, `word3`, and `target`.

    Returns:
        A tuple with:
        - results_df: per-question predictions and correctness flags
        - summary: aggregate totals, coverage, and accuracy
    """
    vocab = model.wv.key_to_index
    results = []

    # TODO: Iterate through each analogy row, check vocabulary coverage, generate a prediction, and mark whether it is correct.
    ############ WRITE YOUR CODE HERE ############

        results.append(
            {
                "type": analogy_row.type,
                "word1": analogy_row.word1,
                "word2": analogy_row.word2,
                "word3": analogy_row.word3,
                "target": analogy_row.target,
                "predicted": predicted,
                "covered": covered,
                "is_correct": is_correct,
            }
        )

    results_df = pd.DataFrame(results)
    covered_df = results_df[results_df["covered"]].copy()

    summary = pd.Series(
        {
            "total_questions": int(len(results_df)),
            "covered_questions": int(len(covered_df)),
            "coverage": covered_df.shape[0] / len(results_df),
            "accuracy_on_covered": covered_df["is_correct"].mean() if not covered_df.empty else 0.0,
        }
    )
    return results_df, summary

In [ ]:
skipgram_analogy_results, skipgram_analogy_summary = evaluate_word_analogies(skipgram_model, msr_df)

print("Skip-gram overall analogy evaluation:")
skipgram_analogy_summary.to_frame(name="value")

skipgram_type_summary = (
    skipgram_analogy_results[skipgram_analogy_results["covered"]]
    .groupby("type")["is_correct"]
    .agg(questions="size", accuracy="mean")
    .sort_values(["accuracy", "questions"], ascending=[False, False])
)

skipgram_type_summary

## Section 3.2: Evaluate Word Analogies (CBOW)

In [ ]:
cbow_analogy_results, cbow_analogy_summary = evaluate_word_analogies(cbow_model, msr_df)

print("CBOW overall analogy evaluation:")
cbow_analogy_summary.to_frame(name="value")

cbow_type_summary = (
    cbow_analogy_results[cbow_analogy_results["covered"]]
    .groupby("type")["is_correct"]
    .agg(questions="size", accuracy="mean")
    .sort_values(["accuracy", "questions"], ascending=[False, False])
)

cbow_type_summary


## Qualitative Check

In [ ]:
# Show 5 random correct covered CBOW analogy predictions.
cbow_covered_examples = cbow_analogy_results[cbow_analogy_results["covered"]].copy()

cbow_correct_examples = cbow_covered_examples[cbow_covered_examples["is_correct"]].copy()
cbow_correct_examples_sample = cbow_correct_examples.sample(
    n=min(5, len(cbow_correct_examples)),
    random_state=42,
)

cbow_correct_examples_sample[["type", "word1", "word2", "word3", "target", "predicted"]]


In [ ]:
# Show 5 random incorrect covered CBOW analogy predictions.
cbow_covered_examples = cbow_analogy_results[cbow_analogy_results["covered"]].copy()

cbow_incorrect_examples = cbow_covered_examples[~cbow_covered_examples["is_correct"]].copy()
cbow_incorrect_examples_sample = cbow_incorrect_examples.sample(
    n=min(5, len(cbow_incorrect_examples)),
    random_state=42,
)

cbow_incorrect_examples_sample[["type", "word1", "word2", "word3", "target", "predicted"]]


In [ ]:
# Qualitative check: Skip-gram types with 0 accuracy (show up to 3 covered cases per type).
skipgram_zero_accuracy_types = skipgram_type_summary[skipgram_type_summary["accuracy"] == 0].index.tolist()

skipgram_zero_accuracy_frames = [
    skipgram_analogy_results[
        (skipgram_analogy_results["covered"]) & (skipgram_analogy_results["type"] == analogy_type)
    ]
    .head(3)
    .assign(model="skipgram", zero_accuracy_type=analogy_type)
    for analogy_type in skipgram_zero_accuracy_types
]

skipgram_zero_accuracy_examples = (
    pd.concat(skipgram_zero_accuracy_frames, ignore_index=True)    
)

skipgram_zero_accuracy_examples[
    ["model", "zero_accuracy_type", "word1", "word2", "word3", "target", "predicted", "is_correct"]
]

In [ ]:
# Qualitative check: CBOW types with 0 accuracy (show up to 3 covered cases per type).
cbow_zero_accuracy_types = cbow_type_summary[cbow_type_summary["accuracy"] == 0].index.tolist()

cbow_zero_accuracy_frames = [
    cbow_analogy_results[
        (cbow_analogy_results["covered"]) & (cbow_analogy_results["type"] == analogy_type)
    ]
    .head(3)
    .assign(model="cbow", zero_accuracy_type=analogy_type)
    for analogy_type in cbow_zero_accuracy_types
]

cbow_zero_accuracy_examples = (
    pd.concat(cbow_zero_accuracy_frames, ignore_index=True)
)

cbow_zero_accuracy_examples[
    ["model", "zero_accuracy_type", "word1", "word2", "word3", "target", "predicted", "is_correct"]
]